In [1]:
"""
01_geometric_mlps.py / 01_geometric_mlps.ipynb

Self-Contained Benchmark: Geometric Projections vs. Standard Normalization in MLPs
Evaluates:
  1. Dense Vision: FashionMNIST (5 layers, hidden dim 256, Cross-Entropy)
  2. Tabular Classification: Synthetic binary classification (4 layers, hidden dim 128, Cross-Entropy)
  3. Continuous Regression: California Housing (4 layers, hidden dim 128, MSE)

Architectures Evaluated:
  - Vanilla: Unnormalized linear layers (sanity-check baseline)
  - BatchNorm: Batch normalization with affine parameters (standard baseline)
  - Hyperspherical: Conical L2 projection on input and weight vectors
  - Equatorial: Zero-trace intra-sample projection (S^{d-2}) with Weight Standardization
"""

import os
import gc
import time
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from sklearn.datasets import make_classification, fetch_california_housing, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -----------------------------------------------------------------------------
# 0. Global Configuration and Hardware Setup
# -----------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS: List[int] = [42, 1337, 2026, 7, 99]
EPS: float = 1e-7

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


# -----------------------------------------------------------------------------
# 1. Geometric Linear Primitives
# -----------------------------------------------------------------------------
class HypersphericalLinear(nn.Module):
    """
    Linear layer with spherical normalization applied to both input features and weights.
    Projects features onto the unit sphere S^{d-1} and rescales by sqrt(in_features).

    Forward Pass:
        x_norm = x / ||x||_2
        w_norm = W / ||W||_2
        y = scale * (x_norm @ w_norm^T)
    """
    def __init__(self, in_features: int, out_features: int, eps: float = 1e-7) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.eps = eps
        
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.orthogonal_(self.weight)
        self.scale = nn.Parameter(torch.tensor(float(np.sqrt(in_features))))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape [B, in_features]
        Returns:
            Output tensor of shape [B, out_features]
        """
        w_norm = F.normalize(self.weight, p=2, dim=1, eps=self.eps)
        x_norm = F.normalize(x, p=2, dim=-1, eps=self.eps)
        return self.scale * F.linear(x_norm, w_norm)


class EquatorialLinear(nn.Module):
    """
    Linear layer combining an intra-sample zero-trace projection onto S^{d-2}
    with Weight Standardization on the parameter matrix.

    Forward Pass:
        x_eq = (x - mean(x)) / std(x)
        w_cent = W - mean(W, dim=1)
        w_norm = w_cent / ||w_cent||_2
        y = scale * (x_eq @ w_norm^T)
    """
    def __init__(self, in_features: int, out_features: int, eps: float = 1e-5) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.eps = eps
        
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.orthogonal_(self.weight)
        self.scale = nn.Parameter(torch.tensor(float(np.sqrt(in_features))))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape [B, in_features]
        Returns:
            Output tensor of shape [B, out_features]
        """
        x_eq = F.layer_norm(x, (self.in_features,), eps=self.eps)
        w_mean = self.weight.mean(dim=1, keepdim=True)
        w_cent = self.weight - w_mean
        w_norm = w_cent / (torch.norm(w_cent, p=2, dim=1, keepdim=True) + self.eps)
        return self.scale * F.linear(x_eq, w_norm)


# -----------------------------------------------------------------------------
# 2. Model Architecture Factory
# -----------------------------------------------------------------------------
def build_mlp(
    layer_type: str,
    in_dim: int,
    out_dim: int,
    hidden_dim: int,
    num_layers: int
) -> nn.Sequential:
    """
    Constructs an MLP with specified depth, hidden dimension, and normalization strategy.

    Args:
        layer_type: One of ['vanilla', 'batchnorm', 'hyperspherical', 'equatorial']
        in_dim: Dimensionality of input features
        out_dim: Dimensionality of output targets/logits
        hidden_dim: Dimensionality of intermediate representations
        num_layers: Total number of linear transformations (including output head)
    """
    layers: List[nn.Module] = []
    curr_dim = in_dim
    
    for _ in range(num_layers - 1):
        if layer_type == "vanilla":
            layers.append(nn.Linear(curr_dim, hidden_dim))
            layers.append(nn.GELU())
        elif layer_type == "batchnorm":
            layers.append(nn.Linear(curr_dim, hidden_dim, bias=False))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.GELU())
        elif layer_type == "hyperspherical":
            layers.append(HypersphericalLinear(curr_dim, hidden_dim))
            layers.append(nn.GELU())
        elif layer_type == "equatorial":
            layers.append(EquatorialLinear(curr_dim, hidden_dim))
            layers.append(nn.GELU())
        else:
            raise ValueError(f"Unknown layer_type: {layer_type}")
        curr_dim = hidden_dim
        
    layers.append(nn.Linear(curr_dim, out_dim))
    return nn.Sequential(*layers)


# -----------------------------------------------------------------------------
# 3. Representational Metric: Stable Rank
# -----------------------------------------------------------------------------
def compute_stable_rank(features: torch.Tensor, eps: float = 1e-7) -> float:
    """
    Computes the stable rank of a centered feature representation matrix:
        srank(H) = ||H||_F^2 / ||H||_2^2 = sum(sigma_i^2) / max(sigma_i)^2

    Args:
        features: Representation matrix of shape [N, D]
        eps: Small numerical constant to avoid division by zero
    Returns:
        Scalar stable rank value
    """
    h_centered = features - features.mean(dim=0, keepdim=True)
    _, s, _ = torch.svd(h_centered)
    srank = (torch.sum(s ** 2) / (torch.max(s) ** 2 + eps)).item()
    return float(srank)


# -----------------------------------------------------------------------------
# 4. Data Preparation Pipeline
# -----------------------------------------------------------------------------
def prepare_datasets() -> Tuple[Subset, DataLoader, DataLoader,
                                Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray],
                                Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:
    """
    Loads and partitions all three evaluation datasets using disjoint train/val/test splits.
    """
    # 4.1 FashionMNIST: 45k train, 5k validation, 10k test
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,))
    ])
    try:
        fmnist_train_full = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
        fmnist_test = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)
    except Exception:
        fmnist_train_full = datasets.FashionMNIST(root="/kaggle/input/fashionmnist", train=True, download=False, transform=transform)
        fmnist_test = datasets.FashionMNIST(root="/kaggle/input/fashionmnist", train=False, download=False, transform=transform)

    fmnist_train_sub = Subset(fmnist_train_full, list(range(45000)))
    fmnist_val_sub = Subset(fmnist_train_full, list(range(45000, 50000)))
    fmnist_val_loader = DataLoader(fmnist_val_sub, batch_size=256, shuffle=False)
    fmnist_test_loader = DataLoader(fmnist_test, batch_size=256, shuffle=False)

    # 4.2 Synthetic Tabular Classification (10,000 samples, 32 features)
    x_c, y_c = make_classification(
        n_samples=10000, n_features=32, n_informative=22, n_redundant=10,
        flip_y=0.03, random_state=42
    )
    x_c_tr, x_c_temp, y_c_tr, y_c_temp = train_test_split(x_c, y_c, test_size=0.30, random_state=42)
    x_c_va, x_c_te, y_c_va, y_c_te = train_test_split(x_c_temp, y_c_temp, test_size=0.50, random_state=42)
    
    scaler_c = StandardScaler()
    x_c_tr = scaler_c.fit_transform(x_c_tr)
    x_c_va = scaler_c.transform(x_c_va)
    x_c_te = scaler_c.transform(x_c_te)
    tabular_cls_data = (x_c_tr, y_c_tr, x_c_va, y_c_va, x_c_te, y_c_te)

    # 4.3 Continuous Regression: California Housing (12,000 samples)
    try:
        reg_data = fetch_california_housing(as_frame=False)
        x_r, y_r = reg_data.data[:12000], reg_data.target[:12000]
    except Exception:
        reg_data = load_diabetes(as_frame=False)
        x_r, y_r = reg_data.data, reg_data.target

    x_r_tr, x_r_temp, y_r_tr, y_r_temp = train_test_split(x_r, y_r, test_size=0.30, random_state=42)
    x_r_va, x_r_te, y_r_va, y_r_te = train_test_split(x_r_temp, y_r_temp, test_size=0.50, random_state=42)

    scaler_rx, scaler_ry = StandardScaler(), StandardScaler()
    x_r_tr = scaler_rx.fit_transform(x_r_tr)
    x_r_va = scaler_rx.transform(x_r_va)
    x_r_te = scaler_rx.transform(x_r_te)
    y_r_tr = scaler_ry.fit_transform(y_r_tr.reshape(-1, 1)).flatten()
    y_r_va = scaler_ry.transform(y_r_va.reshape(-1, 1)).flatten()
    y_r_te = scaler_ry.transform(y_r_te.reshape(-1, 1)).flatten()
    tabular_reg_data = (x_r_tr, y_r_tr, x_r_va, y_r_va, x_r_te, y_r_te)

    return fmnist_train_sub, fmnist_val_loader, fmnist_test_loader, tabular_cls_data, tabular_reg_data


# -----------------------------------------------------------------------------
# 5. Benchmark Execution Loop
# -----------------------------------------------------------------------------
def run_benchmark() -> None:
    (
        fmnist_train_sub,
        fm_val_loader,
        fm_test_loader,
        (x_c_tr, y_c_tr, x_c_va, y_c_va, x_c_te, y_c_te),
        (x_r_tr, y_r_tr, x_r_va, y_r_va, x_r_te, y_r_te)
    ) = prepare_datasets()

    c_test_loader = DataLoader(
        TensorDataset(torch.tensor(x_c_te, dtype=torch.float32), torch.tensor(y_c_te, dtype=torch.long)),
        batch_size=256, shuffle=False
    )
    r_test_loader = DataLoader(
        TensorDataset(torch.tensor(x_r_te, dtype=torch.float32), torch.tensor(y_r_te, dtype=torch.float32).unsqueeze(1)),
        batch_size=256, shuffle=False
    )

    models_to_test = [
        ("Vanilla", "vanilla"),
        ("BatchNorm", "batchnorm"),
        ("Hyperspherical", "hyperspherical"),
        ("Equatorial", "equatorial")
    ]

    results_fmnist = {m[0]: {"acc": [], "srank": []} for m in models_to_test}
    results_cls = {m[0]: {"acc": [], "srank": []} for m in models_to_test}
    results_reg = {m[0]: {"mse": [], "srank": []} for m in models_to_test}

    print("=" * 115)
    print(f"[INFO] Multi-Seed MLP Benchmark | Device: {DEVICE} | Seeds: {len(SEEDS)}")
    print("=" * 115)

    # -------------------------------------------------------------------------
    # Task 1: FashionMNIST
    # -------------------------------------------------------------------------
    print("\n[INFO] Starting Task 1: FashionMNIST (5 Layers, Dim 256, 6 Epochs)")
    for seed in SEEDS:
        g = torch.Generator().manual_seed(seed)
        fm_tr_loader = DataLoader(fmnist_train_sub, batch_size=128, shuffle=True, drop_last=True, generator=g)

        for name, layer_type in models_to_test:
            torch.manual_seed(seed)
            model = build_mlp(layer_type, in_dim=784, out_dim=10, hidden_dim=256, num_layers=5).to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
            criterion = nn.CrossEntropyLoss()

            for _ in range(6):
                model.train()
                for xb, yb in fm_tr_loader:
                    xb, yb = xb.view(xb.size(0), -1).to(DEVICE), yb.to(DEVICE)
                    optimizer.zero_grad()
                    criterion(model(xb), yb).backward()
                    optimizer.step()

            model.eval()
            correct, total = 0, 0
            features_list: List[torch.Tensor] = []
            with torch.no_grad():
                for xb, yb in fm_test_loader:
                    xb, yb = xb.view(xb.size(0), -1).to(DEVICE), yb.to(DEVICE)
                    feats = model[:-1](xb)
                    features_list.append(feats.cpu())
                    correct += (model[-1](feats).argmax(dim=-1) == yb).sum().item()
                    total += yb.size(0)

            acc = (correct / total) * 100.0
            sr = compute_stable_rank(torch.cat(features_list, dim=0))
            results_fmnist[name]["acc"].append(acc)
            results_fmnist[name]["srank"].append(sr)

            del model, optimizer
            torch.cuda.empty_cache()

        print(f"[INFO] [Task 1 - FashionMNIST] Completed Seed {seed}")

    # -------------------------------------------------------------------------
    # Task 2: Tabular Classification
    # -------------------------------------------------------------------------
    print("\n[INFO] Starting Task 2: Tabular Classification (4 Layers, Dim 128, 8 Epochs)")
    for seed in SEEDS:
        tr_loader = DataLoader(
            TensorDataset(torch.tensor(x_c_tr, dtype=torch.float32), torch.tensor(y_c_tr, dtype=torch.long)),
            batch_size=64, shuffle=True
        )

        for name, layer_type in models_to_test:
            torch.manual_seed(seed)
            model = build_mlp(layer_type, in_dim=32, out_dim=2, hidden_dim=128, num_layers=4).to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
            criterion = nn.CrossEntropyLoss()

            for _ in range(8):
                model.train()
                for xb, yb in tr_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    optimizer.zero_grad()
                    criterion(model(xb), yb).backward()
                    optimizer.step()

            model.eval()
            correct, total = 0, 0
            features_list = []
            with torch.no_grad():
                for xb, yb in c_test_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    feats = model[:-1](xb)
                    features_list.append(feats.cpu())
                    correct += (model[-1](feats).argmax(dim=-1) == yb).sum().item()
                    total += yb.size(0)

            acc = (correct / total) * 100.0
            sr = compute_stable_rank(torch.cat(features_list, dim=0))
            results_cls[name]["acc"].append(acc)
            results_cls[name]["srank"].append(sr)

            del model, optimizer
            torch.cuda.empty_cache()

        print(f"[INFO] [Task 2 - Tabular Classification] Completed Seed {seed}")

    # -------------------------------------------------------------------------
    # Task 3: Continuous Regression
    # -------------------------------------------------------------------------
    print("\n[INFO] Starting Task 3: California Housing Regression (4 Layers, Dim 128, 12 Epochs)")
    for seed in SEEDS:
        tr_loader = DataLoader(
            TensorDataset(torch.tensor(x_r_tr, dtype=torch.float32), torch.tensor(y_r_tr, dtype=torch.float32).unsqueeze(1)),
            batch_size=64, shuffle=True
        )

        for name, layer_type in models_to_test:
            torch.manual_seed(seed)
            model = build_mlp(layer_type, in_dim=x_r_tr.shape[1], out_dim=1, hidden_dim=128, num_layers=4).to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
            criterion = nn.MSELoss()

            for _ in range(12):
                model.train()
                for xb, yb in tr_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    optimizer.zero_grad()
                    criterion(model(xb), yb).backward()
                    optimizer.step()

            model.eval()
            total_mse, total_samples = 0.0, 0
            features_list = []
            with torch.no_grad():
                for xb, yb in r_test_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    feats = model[:-1](xb)
                    features_list.append(feats.cpu())
                    preds = model[-1](feats)
                    total_mse += criterion(preds, yb).item() * yb.size(0)
                    total_samples += yb.size(0)

            mse_val = total_mse / total_samples
            sr = compute_stable_rank(torch.cat(features_list, dim=0))
            results_reg[name]["mse"].append(mse_val)
            results_reg[name]["srank"].append(sr)

            del model, optimizer
            torch.cuda.empty_cache()

        print(f"[INFO] [Task 3 - Continuous Regression] Completed Seed {seed}")

    # -------------------------------------------------------------------------
    # 6. Tabulated Performance Reports
    # -------------------------------------------------------------------------
    print("\n" + "=" * 115)
    print("EMPIRICAL PERFORMANCE SUMMARY (5 SEEDS, MEAN +/- STD)")
    print("=" * 115)
    print(f"{'ARCHITECTURE':<20} | {'FASHION-MNIST (ACC %)':<24} | {'TABULAR CLS (ACC %)':<22} | {'REGRESSION (MSE)'}")
    print("-" * 115)
    for name, _ in models_to_test:
        fm_m, fm_s = np.mean(results_fmnist[name]["acc"]), np.std(results_fmnist[name]["acc"])
        cls_m, cls_s = np.mean(results_cls[name]["acc"]), np.std(results_cls[name]["acc"])
        reg_m, reg_s = np.mean(results_reg[name]["mse"]), np.std(results_reg[name]["mse"])
        print(f"{name:<20} | {fm_m:>9.2f}% +/- {fm_s:<10.2f} | {cls_m:>8.2f}% +/- {cls_s:<9.2f} | {reg_m:>8.4f} +/- {reg_s:<8.4f}")
    print("=" * 115)

    print("\n" + "=" * 115)
    print("REPRESENTATIONAL HEALTH: STABLE RANK (5 SEEDS, MEAN +/- STD)")
    print("=" * 115)
    print(f"{'ARCHITECTURE':<20} | {'SRANK FASHION (256D)':<24} | {'SRANK CLS (128D)':<22} | {'SRANK REG (128D)'}")
    print("-" * 115)
    for name, _ in models_to_test:
        fm_sr_m, fm_sr_s = np.mean(results_fmnist[name]["srank"]), np.std(results_fmnist[name]["srank"])
        cls_sr_m, cls_sr_s = np.mean(results_cls[name]["srank"]), np.std(results_cls[name]["srank"])
        reg_sr_m, reg_sr_s = np.mean(results_reg[name]["srank"]), np.std(results_reg[name]["srank"])
        print(f"{name:<20} | {fm_sr_m:>9.2f} +/- {fm_sr_s:<12.2f} | {cls_sr_m:>8.2f} +/- {cls_sr_s:<11.2f} | {reg_sr_m:>8.2f} +/- {reg_sr_s:<8.2f}")
    print("=" * 115)


if __name__ == "__main__":
    run_benchmark()

100%|██████████| 26.4M/26.4M [00:00<00:00, 115MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.76MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 54.9MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.4MB/s]


[INFO] Multi-Seed MLP Benchmark | Device: cuda | Seeds: 5

[INFO] Starting Task 1: FashionMNIST (5 Layers, Dim 256, 6 Epochs)
[INFO] [Task 1 - FashionMNIST] Completed Seed 42
[INFO] [Task 1 - FashionMNIST] Completed Seed 1337
[INFO] [Task 1 - FashionMNIST] Completed Seed 2026
[INFO] [Task 1 - FashionMNIST] Completed Seed 7
[INFO] [Task 1 - FashionMNIST] Completed Seed 99

[INFO] Starting Task 2: Tabular Classification (4 Layers, Dim 128, 8 Epochs)
[INFO] [Task 2 - Tabular Classification] Completed Seed 42
[INFO] [Task 2 - Tabular Classification] Completed Seed 1337
[INFO] [Task 2 - Tabular Classification] Completed Seed 2026
[INFO] [Task 2 - Tabular Classification] Completed Seed 7
[INFO] [Task 2 - Tabular Classification] Completed Seed 99

[INFO] Starting Task 3: California Housing Regression (4 Layers, Dim 128, 12 Epochs)
[INFO] [Task 3 - Continuous Regression] Completed Seed 42
[INFO] [Task 3 - Continuous Regression] Completed Seed 1337
[INFO] [Task 3 - Continuous Regression] Comple